In [5]:
# we use the dataset https://github.com/gaia-solutions-on-demand/DFireDataset/blob/master/figures/dfire_examples.png as a training set
# we also annotated manually using roboflow part of the dataset from "defi1" to serve as a test set 



In [5]:
from ruamel.yaml import YAML
from ultralytics.utils import DEFAULT_CFG_DICT

yaml = YAML(typ='safe')
with open('configs/train.yaml', 'r') as f:
    cfg = yaml.load(f)

print(f"Configured dataset fraction: {cfg['fraction']:.0%}")
print(f"Ultralytics default fraction: {DEFAULT_CFG_DICT['fraction']}")

import argparse
from ultralytics import YOLO
from ruamel.yaml import YAML
import torch
import os
import random   
import numpy as np

def set_seed(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def load_yaml(path):
    yaml = YAML(typ='safe')
    with open(path, 'r') as f:
        return yaml.load(f)
    
config = "configs/train.yaml"
seed=42
device="0"

set_seed(seed)
train_cfg = load_yaml(config)

# Determine device
if device is not None:
    device = device
else:
    device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the model defined in the training config file
model = YOLO(train_cfg['model'])

# Train the model
results = model.train(device=device, **train_cfg)
print(results)




Configured dataset fraction: 100%
Ultralytics default fraction: 1.0
Using device: 0
Ultralytics 8.4.51 🚀 Python-3.13.5 torch-2.12.0 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 0
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [5]:
from ultralytics import YOLO

# Load a model
model = YOLO("/Users/youss/Desktop/best.pt")  # pretrained YOLO26n model

# Run batched inference on a list of images
results = model(["/Users/youss/Desktop/fire_test_1013.jpg"])  # return a list of Results objects

# Process results list
for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen
    result.save(filename="result.jpg")  # save to disk


0: 640x640 1 fire, 1 smoke, 104.1ms
Speed: 13.9ms preprocess, 104.1ms inference, 17.7ms postprocess per image at shape (1, 3, 640, 640)


In [6]:
from ultralytics import YOLO

# Load your fine-tuned model
model = YOLO("/Users/youss/Desktop/best.pt")

# Correct class names
correct_names = {
    0: "smoke",
    1: "fire",
}

model.model.names = correct_names

# Run prediction
results = model(["/Users/youss/Desktop/fire_test_1013.jpg"])

# Process results
for i, result in enumerate(results):
    result.names = correct_names

    boxes = result.boxes

    for box in boxes:
        class_id = int(box.cls[0])
        class_name = correct_names[class_id]
        confidence = float(box.conf[0])

        print(class_name, confidence)

    result.show()
    result.save(filename=f"result_{i}.jpg")


0: 640x640 1 smoke, 1 fire, 99.8ms
Speed: 18.1ms preprocess, 99.8ms inference, 26.1ms postprocess per image at shape (1, 3, 640, 640)
smoke 0.855006217956543
fire 0.7579503655433655


In [4]:
import cv2
from ultralytics import YOLO

model = YOLO("/Users/youss/Desktop/best.pt")
correct_names = {0: "smoke", 1: "fire"}
model.model.names = correct_names

video_path = "/Users/youss/Desktop/Vidéo Feu.mp4"
cap = cv2.VideoCapture(video_path)

# Setup output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("/Users/youss/Desktop/result_video.mp4", fourcc, fps, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    for result in results:
        result.names = correct_names
        for box in result.boxes:
            class_name = correct_names[int(box.cls[0])]
            confidence = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(frame, f"{class_name} {confidence:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    out.write(frame)

cap.release()
out.release()
print("Vidéo sauvegardée : /Users/youss/Desktop/result_video.mp4")


0: 384x640 1 fire, 88.1ms
Speed: 16.1ms preprocess, 88.1ms inference, 24.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 44.0ms
Speed: 2.3ms preprocess, 44.0ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 39.8ms
Speed: 2.2ms preprocess, 39.8ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 smoke, 1 fire, 40.2ms
Speed: 2.1ms preprocess, 40.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 smoke, 1 fire, 39.6ms
Speed: 1.6ms preprocess, 39.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fire, 38.8ms
Speed: 1.7ms preprocess, 38.8ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 smoke, 3 fires, 40.7ms
Speed: 1.6ms preprocess, 40.7ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fire, 39.8ms
Speed: 2.4ms preprocess, 39.8ms inference, 0.3ms postproc

In [ ]:
import os
import cv2
import numpy as np
import torch
import quantus
from ultralytics import YOLO

MODEL_PATH = "/Users/youss/Desktop/fire-detection/yolo26_fire_detection7/weights/best.pt"
IMAGES_DIR = "/Users/youss/Desktop/D-Fire/test/images"
LABELS_DIR = "/Users/youss/Desktop/D-Fire/test/labels"
IMG_SIZE = 640

model = YOLO(MODEL_PATH)
torch_model = model.model
torch_model.train()

def load_image(path):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def yolo_label_to_mask(label_path, img_h, img_w):
    mask = np.zeros((img_h, img_w), dtype=np.float32)
    if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
        return mask
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, w, h = map(float, parts)
            x1 = int((cx - w/2) * img_w)
            y1 = int((cy - h/2) * img_h)
            x2 = int((cx + w/2) * img_w)
            y2 = int((cy + h/2) * img_h)
            mask[y1:y2, x1:x2] = 1.0
    return mask

target_layer = torch_model.model[9]

def get_gradcam_attribution(img_rgb, target_layer):
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    tensor = torch.from_numpy(img_resized).permute(2,0,1).float().unsqueeze(0) / 255.0

    activations = []
    gradients = []

    def forward_hook(module, input, output):
        activations.append(output.detach())

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0].detach())

    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)

    with torch.enable_grad():
        tensor = tensor.requires_grad_(True)
        output = torch_model(tensor)
        if isinstance(output, (tuple, list)):
            output = output[0]
        if isinstance(output, dict):
            output = list(output.values())[0]
        loss = output.max()
        loss.backward()

    h1.remove()
    h2.remove()

    act = activations[0].squeeze(0)
    grad = gradients[0].squeeze(0)
    weights = grad.mean(dim=(1, 2))
    cam = (weights[:, None, None] * act).sum(0)
    cam = torch.relu(cam).numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam

img_files_all = set(f.replace('.jpg','') for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg'))
lbl_files_all = set(f.replace('.txt','') for f in os.listdir(LABELS_DIR))
common = img_files_all & lbl_files_all
img_files = [f + '.jpg' for f in common][:500]

images_list, attributions_list, masks_list = [], [], []

for fname in img_files:
    img_path = os.path.join(IMAGES_DIR, fname)
    label_path = os.path.join(LABELS_DIR, fname.replace('.jpg', '.txt'))

    img = load_image(img_path)
    if img is None:
        continue
    mask = yolo_label_to_mask(label_path, IMG_SIZE, IMG_SIZE)
    if mask.sum() == 0:
        continue

    attribution = get_gradcam_attribution(img, target_layer)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    images_list.append(img_resized.transpose(2,0,1) / 255.0)
    attributions_list.append(attribution[np.newaxis])
    masks_list.append(mask[np.newaxis])

x_batch = np.array(images_list)
a_batch = np.array(attributions_list)
s_batch = np.array(masks_list).astype(bool)

print(f"x_batch: {x_batch.shape}")
print(f"a_batch: {a_batch.shape}")
print(f"s_batch: {s_batch.shape}")
print(f"Images avec annotations: {len(x_batch)}")

pointing_game = quantus.PointingGame(normalise=True, abs=True)
attr_loc = quantus.AttributionLocalisation(normalise=True, abs=True)
top_k = quantus.TopKIntersection(normalise=True, abs=True)

y_batch = np.zeros(len(x_batch), dtype=int)

pg_scores = pointing_game(model=None, x_batch=x_batch, y_batch=y_batch, a_batch=a_batch, s_batch=s_batch, channel_first=True)
al_scores = attr_loc(model=None, x_batch=x_batch, y_batch=y_batch, a_batch=a_batch, s_batch=s_batch, channel_first=True)
tk_scores = top_k(model=None, x_batch=x_batch, y_batch=y_batch, a_batch=a_batch, s_batch=s_batch, channel_first=True)

print(f"Pointing Game:            {np.mean(pg_scores):.3f}")
print(f"Attribution Localisation: {np.mean(al_scores):.3f}")
print(f"Top-K Intersection:       {np.mean(tk_scores):.3f}")

/var/folders/zm/t4ysddjs4ln1sxxcd0rn26t40000gn/T/ipykernel_16059/3922909985.py:67: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


x_batch: (18, 3, 640, 640)
a_batch: (18, 1, 640, 640)
s_batch: (18, 1, 640, 640)
Images avec annotations: 18


AssertionError: The elements in the attribution vector are all equal to zero, which may cause inconsistent results since many metrics rely on ordering. Recompute the explanations.

In [25]:
img_files = set(f.replace('.jpg','') for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg'))
lbl_files = set(f.replace('.txt','') for f in os.listdir(LABELS_DIR))
common = img_files & lbl_files
print(f"Correspondances: {len(common)}")

Correspondances: 4306
